In [1]:
!pip -q install sentence-transformers faiss-cpu pandas numpy

In [2]:
import pandas as pd
import numpy as np
import faiss
import pickle

from sentence_transformers import SentenceTransformer
from IPython.display import display

In [3]:
input_path = "./rmit_assessment_support_scraped_cleaned.csv"

df = pd.read_csv(input_path)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (64, 11)


,source_id,category,title,section,content,source_url,source_type,collection_date,effective_date,content_length,document_version
0,EXT-01,Extension,Extensions,If you can't submit an assessment on time due ...,If you are prevented from submitting an assess...,https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-09-09,Not specified,541,Current webpage
1,EXT-01,Extension,Extensions,Assessments eligible for an extension,You can apply for an extension for assessments...,https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-09-09,Not specified,316,Current webpage
2,EXT-01,Extension,Extensions,How to apply,You must apply at least one working day before...,https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-09-09,Not specified,1320,Current webpage
3,EXT-01,Extension,Extensions,False documents and misleading information,"Creating, submitting or using fraudulent docum...",https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-09-09,Not specified,577,Current webpage
4,SC-01,Special Consideration,Special consideration,Special consideration,If unexpected circumstances beyond your contro...,https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-09-09,Not specified,315,Current webpage


In [4]:
print("Columns:")
print(df.columns.tolist())

print("\nNumber of records:", len(df))

print("\nRecords by source:")
print(df["source_id"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())

Columns:
['source_id', 'category', 'title', 'section', 'content', 'source_url', 'source_type', 'collection_date', 'effective_date', 'content_length', 'document_version']

Number of records: 64

Records by source:
source_id
EAA-02    34
POL-01    16
SC-01      8
EXT-01     4
EAA-01     2
Name: count, dtype: int64

Missing values:
source_id           0
category            0
title               0
section             0
content             0
source_url          0
source_type         0
collection_date     0
effective_date      0
content_length      0
document_version    0
dtype: int64


In [5]:
print("Example content:\n")
print(df.loc[0, "content"])

Example content:

If you are prevented from submitting an assessment on time due to unexpected/unforeseen circumstances that are outside your control, and are of a short-term nature, you may apply in advance for an extension to the due date of up to seven calendar days . If you need an extension of more than seven days , you must apply for special consideration . However, if you have an equitable assessment arrangement which allows for the negotiation of submission dates with academic/teaching staff, extensions of more than seven days may be considered.


In [6]:
df["word_count"] = df["content"].str.split().str.len()

print("Minimum words:", df["word_count"].min())
print("Maximum words:", df["word_count"].max())
print("Average words:", round(df["word_count"].mean(), 2))

Minimum words: 14
Maximum words: 1787
Average words: 183.55


In [7]:
def create_chunks(text, chunk_size=500, overlap=100):
    words = text.split()

    chunks = []
    start = 0

    while start < len(words):

        end = min(start + chunk_size, len(words))

        chunk = " ".join(words[start:end])

        chunks.append(chunk)

        if end == len(words):
            break

        start = end - overlap

    return chunks

In [8]:
test_text = df.loc[0, "content"]

test_chunks = create_chunks(
    test_text,
    chunk_size=500,
    overlap=100
)

print("Original words:", len(test_text.split()))
print("Number of chunks:", len(test_chunks))

for i, chunk in enumerate(test_chunks):
    print(f"\n--- Chunk {i+1} ---")
    print("Words:", len(chunk.split()))
    print(chunk[:500])

Original words: 90
Number of chunks: 1

--- Chunk 1 ---
Words: 90
If you are prevented from submitting an assessment on time due to unexpected/unforeseen circumstances that are outside your control, and are of a short-term nature, you may apply in advance for an extension to the due date of up to seven calendar days . If you need an extension of more than seven days , you must apply for special consideration . However, if you have an equitable assessment arrangement which allows for the negotiation of submission dates with academic/teaching staff, extensions o


In [9]:
chunk_records = []

for _, row in df.iterrows():

    chunks = create_chunks(
        row["content"],
        chunk_size=500,
        overlap=100
    )

    for chunk_id, chunk in enumerate(chunks):

        chunk_records.append({
            "chunk_id": f"{row['source_id']}_{len(chunk_records)}",
            "source_id": row["source_id"],
            "category": row["category"],
            "title": row["title"],
            "section": row["section"],
            "content": chunk,
            "source_url": row["source_url"],
            "source_type": row["source_type"],
            "collection_date": row["collection_date"],
            "effective_date": row["effective_date"],
            "document_version": row["document_version"],
            "chunk_number": chunk_id + 1,
            "total_chunks": len(chunks)
        })

chunks_df = pd.DataFrame(chunk_records)

print("Original records:", len(df))
print("Total chunks:", len(chunks_df))

display(chunks_df.head())

Original records: 64
Total chunks: 71


,chunk_id,source_id,category,title,section,content,source_url,source_type,collection_date,effective_date,document_version,chunk_number,total_chunks
0,EXT-01_0,EXT-01,Extension,Extensions,If you can't submit an assessment on time due ...,If you are prevented from submitting an assess...,https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-09-09,Not specified,Current webpage,1,1
1,EXT-01_1,EXT-01,Extension,Extensions,Assessments eligible for an extension,You can apply for an extension for assessments...,https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-09-09,Not specified,Current webpage,1,1
2,EXT-01_2,EXT-01,Extension,Extensions,How to apply,You must apply at least one working day before...,https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-09-09,Not specified,Current webpage,1,1
3,EXT-01_3,EXT-01,Extension,Extensions,False documents and misleading information,"Creating, submitting or using fraudulent docum...",https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-09-09,Not specified,Current webpage,1,1
4,SC-01_4,SC-01,Special Consideration,Special consideration,Special consideration,If unexpected circumstances beyond your contro...,https://www.rmit.edu.au/students/my-course/ass...,webpage,2026-09-09,Not specified,Current webpage,1,1


In [10]:
chunks_df["chunk_word_count"] = (
    chunks_df["content"]
    .str.split()
    .str.len()
)

print("Chunk statistics:")
print(chunks_df["chunk_word_count"].describe())

Chunk statistics:
count     71.000000
mean     175.309859
std      160.716751
min       14.000000
25%       54.500000
50%       92.000000
75%      256.000000
max      500.000000
Name: chunk_word_count, dtype: float64


In [11]:
print("\nChunks by source:")
print(chunks_df["source_id"].value_counts())


Chunks by source:
source_id
EAA-02    35
POL-01    17
SC-01     13
EXT-01     4
EAA-01     2
Name: count, dtype: int64


In [14]:
print("\nChunks by category:")
print(
    chunks_df["category"].value_counts()
)


Chunks by category:
category
Equitable Assessment Arrangements    37
Policy                               17
Special Consideration                13
Extension                             4
Name: count, dtype: int64


In [12]:
short_chunks = chunks_df[
    chunks_df["chunk_word_count"] < 50
]

print("Chunks under 50 words:", len(short_chunks))

display(
    short_chunks[
        [
            "chunk_id",
            "source_id",
            "section",
            "chunk_number",
            "chunk_word_count",
            "content"
        ]
    ]
)

Chunks under 50 words: 15


,chunk_id,source_id,section,chunk_number,chunk_word_count,content
4,SC-01_4,SC-01,Special consideration,1,46,If unexpected circumstances beyond your contro...
5,SC-01_5,SC-01,If unexpected circumstances beyond your contro...,1,14,What is special consideration? Eligibility How...
19,EAA-02_19,EAA-02,Should I disclose my disability/condition to R...,1,29,You only need to tell the Equitable Learning S...
23,EAA-02_23,EAA-02,My Equitable Learning Plan has expired or will...,1,31,"If your Equitable Learning Plan expires, you c..."
24,EAA-02_24,EAA-02,I didn’t need ELS support or an Equitable Lear...,1,36,"Yes, absolutely. You can request ELS support a..."
26,EAA-02_26,EAA-02,Top tips for students seeking ELS support,1,18,"Equitable Learning Advisor, Sarah, shares her ..."
36,EAA-02_36,EAA-02,Can I request more than one extension for the ...,1,39,"No, if you have the Equitable Assessment Arran..."
38,EAA-02_38,EAA-02,I've requested an extension via email and I ha...,1,23,"If you can't get a response from your teacher,..."
39,EAA-02_39,EAA-02,My School has asked me to fill out a form when...,1,46,If you have Equitable Assessment Arrangement f...
42,EAA-02_42,EAA-02,Do my Equitable Assessment Arrangements apply ...,1,31,"Yes. They apply to exams, in-class and online ..."


In [16]:
chunks_path = "./rmit_assessment_support_chunks.csv"

chunks_df.to_csv(
    chunks_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", chunks_path)
print("Total chunks:", len(chunks_df))

Saved: ./rmit_assessment_support_chunks.csv
Total chunks: 71


<h1>Loading the embedding model</h1>

we will use `all-MiniLM-L6-v2` model because It is a lightweight Sentence Transformers model and produces: 384-dimensional embeddings

In [17]:
model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


In [18]:
texts = chunks_df["content"].tolist()

print("Number of texts to embed:", len(texts))

Number of texts to embed: 71


In [19]:
embeddings = model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embedding shape: (71, 384)


In [20]:
faiss.normalize_L2(embeddings)

print("Embeddings normalized successfully.")

Embeddings normalized successfully.


In [21]:
embedding_dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(
    embedding_dimension
)

index.add(embeddings)

print("FAISS index created successfully.")
print("Number of vectors:", index.ntotal)
print("Embedding dimension:", embedding_dimension)

FAISS index created successfully.
Number of vectors: 71
Embedding dimension: 384


In [22]:
index_path = "./rmit_assessment_support.faiss"

faiss.write_index(
    index,
    index_path
)

print("FAISS index saved successfully.")
print("Path:", index_path)

FAISS index saved successfully.
Path: ./rmit_assessment_support.faiss


In [23]:
metadata = chunks_df[
    [
        "chunk_id",
        "source_id",
        "category",
        "title",
        "section",
        "content",
        "source_url",
        "source_type",
        "collection_date",
        "effective_date",
        "document_version",
        "chunk_number",
        "total_chunks"
    ]
].to_dict("records")

In [24]:
metadata_path = "./rmit_assessment_support_metadata.pkl"

with open(metadata_path, "wb") as f:
    pickle.dump(metadata, f)

print("Metadata saved successfully.")
print("Path:", metadata_path)

Metadata saved successfully.
Path: ./rmit_assessment_support_metadata.pkl


In [25]:
def retrieve_documents(query, k=5):

    # Convert the student's question into a vector
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    )

    # Normalize the question vector
    faiss.normalize_L2(query_embedding)

    # Search FAISS for the k most similar chunks
    scores, indices = index.search(
        query_embedding,
        k
    )

    # Store the retrieved chunks and their metadata
    results = []

    for score, idx in zip(scores[0], indices[0]):

        result = metadata[idx].copy()
        result["similarity_score"] = float(score)

        results.append(result)

    return pd.DataFrame(results)

In [27]:
query = "How do I apply for an extension for an assessment?"

results = retrieve_documents(
    query,
    k=5
)

display(
    results[
        [
            "similarity_score",
            "source_id",
            "category",
            "section",
            "content"
        ]
    ]
)

,similarity_score,source_id,category,section,content
0,0.711274,EAA-02,Equitable Assessment Arrangements,How do I ask for an extension?,If you have the Equitable Assessment Arrangeme...
1,0.664620,EAA-02,Equitable Assessment Arrangements,How many days can I request for an extension (...,If you have the Equitable Assessment Arrangeme...
2,0.647993,EXT-01,Extension,Assessments eligible for an extension,You can apply for an extension for assessments...
3,0.645070,EXT-01,Extension,If you can't submit an assessment on time due ...,If you are prevented from submitting an assess...
4,0.640972,POL-01,Policy,Extensions,(34) Extensions are available for unforeseen c...


In [28]:
query = "When should I apply for special consideration?"

results = retrieve_documents(
    query,
    k=5
)

display(
    results[
        [
            "similarity_score",
            "source_id",
            "category",
            "section",
            "content"
        ]
    ]
)

,similarity_score,source_id,category,section,content
0,0.616606,SC-01,Special Consideration,Special consideration,If unexpected circumstances beyond your contro...
1,0.537247,POL-01,Policy,Assessment Flexibility,"(33) The purpose of special consideration, ext..."
2,0.517705,EAA-02,Equitable Assessment Arrangements,My teacher has directed me to apply for specia...,"If it is your first extension, you do not need..."
3,0.488250,SC-01,Special Consideration,False documents and misleading information,"domestic arrangements, e.g. homelessness, evic..."
4,0.485910,SC-01,Special Consideration,If unexpected circumstances beyond your contro...,What is special consideration? Eligibility How...


In [29]:
query = "What are equitable assessment arrangements and who can receive them?"

results = retrieve_documents(
    query,
    k=5
)

display(
    results[
        [
            "similarity_score",
            "source_id",
            "category",
            "section",
            "content"
        ]
    ]
)

,similarity_score,source_id,category,section,content
0,0.761908,POL-01,Policy,Equitable Assessment Arrangements,(54) Equitable assessment arrangements provide...
1,0.703649,EAA-01,Equitable Assessment Arrangements,What is an EAA?,An Equitable Assessment Arrangement (EAA) is a...
2,0.598752,POL-01,Policy,Section 1 - Purpose,(1) To ensure: Relevant and authentic assessme...
3,0.540559,POL-01,Policy,Students and Assessment,(15) Course guides specify all assessment requ...
4,0.537763,EAA-02,Equitable Assessment Arrangements,How many days can I request for an extension (...,If you have the Equitable Assessment Arrangeme...


In [30]:
query = "How many days can an assessment extension be approved for?"

results = retrieve_documents(
    query,
    k=5
)

display(
    results[
        [
            "similarity_score",
            "source_id",
            "category",
            "section",
            "content"
        ]
    ]
)

,similarity_score,source_id,category,section,content
0,0.835225,EXT-01,Extension,If you can't submit an assessment on time due ...,If you are prevented from submitting an assess...
1,0.789507,POL-01,Policy,Extensions,(34) Extensions are available for unforeseen c...
2,0.742802,EAA-02,Equitable Assessment Arrangements,How do I ask for an extension?,If you have the Equitable Assessment Arrangeme...
3,0.706117,EAA-02,Equitable Assessment Arrangements,How many days can I request for an extension (...,If you have the Equitable Assessment Arrangeme...
4,0.691605,EAA-02,Equitable Assessment Arrangements,Can I request an extension for a group assessm...,Extensions in your ELP only apply to individua...


In [31]:
evaluation_questions = [
    {
        "question": "How do I apply for an assessment extension?",
        "expected_sources": ["EXT-01", "POL-01"]
    },
    {
        "question": "When should I apply for special consideration?",
        "expected_sources": ["SC-01", "POL-01"]
    },
    {
        "question": "What are equitable assessment arrangements and who can receive them?",
        "expected_sources": ["EAA-01", "EAA-02", "POL-01"]
    },
    {
        "question": "How many days can an assessment extension be approved for?",
        "expected_sources": ["EXT-01", "POL-01"]
    }
]

print("Number of evaluation questions:", len(evaluation_questions))

Number of evaluation questions: 4


In [32]:
evaluation_results = []

for item in evaluation_questions:

    results = retrieve_documents(
        item["question"],
        k=5
    )

    retrieved_sources = results["source_id"].tolist()

    expected_sources = item["expected_sources"]

    top1_hit = any(
        source in expected_sources
        for source in retrieved_sources[:1]
    )

    top3_hit = any(
        source in expected_sources
        for source in retrieved_sources[:3]
    )

    top5_hit = any(
        source in expected_sources
        for source in retrieved_sources[:5]
    )

    evaluation_results.append({
        "question": item["question"],
        "expected_sources": ", ".join(expected_sources),
        "retrieved_top5": ", ".join(retrieved_sources),
        "top1_hit": top1_hit,
        "top3_hit": top3_hit,
        "top5_hit": top5_hit
    })

evaluation_df = pd.DataFrame(evaluation_results)

display(evaluation_df)

,question,expected_sources,retrieved_top5,top1_hit,top3_hit,top5_hit
0,How do I apply for an assessment extension?,"EXT-01, POL-01","EAA-02, EXT-01, EAA-02, EXT-01, POL-01",False,True,True
1,When should I apply for special consideration?,"SC-01, POL-01","SC-01, POL-01, EAA-02, SC-01, SC-01",True,True,True
2,What are equitable assessment arrangements and...,"EAA-01, EAA-02, POL-01","POL-01, EAA-01, POL-01, POL-01, EAA-02",True,True,True
3,How many days can an assessment extension be a...,"EXT-01, POL-01","EXT-01, POL-01, EAA-02, EAA-02, EAA-02",True,True,True


In [33]:
top1_accuracy = evaluation_df["top1_hit"].mean()
top3_accuracy = evaluation_df["top3_hit"].mean()
top5_accuracy = evaluation_df["top5_hit"].mean()

print(f"Top-1 retrieval accuracy: {top1_accuracy:.2%}")
print(f"Top-3 retrieval accuracy: {top3_accuracy:.2%}")
print(f"Top-5 retrieval accuracy: {top5_accuracy:.2%}")

Top-1 retrieval accuracy: 75.00%
Top-3 retrieval accuracy: 100.00%
Top-5 retrieval accuracy: 100.00%


In [34]:
evaluation_path = "./rmit_retrieval_initial_evaluation.csv"

evaluation_df.to_csv(
    evaluation_path,
    index=False,
    encoding="utf-8-sig"
)

print("Evaluation results saved successfully.")
print("Path:", evaluation_path)

Evaluation results saved successfully.
Path: ./rmit_retrieval_initial_evaluation.csv


In [35]:
import os

files_to_check = [
    "./rmit_assessment_support_chunks.csv",
    "./rmit_assessment_support.faiss",
    "./rmit_assessment_support_metadata.pkl",
    "./rmit_retrieval_initial_evaluation.csv"
]

print("Notebook 2 output files:\n")

for file_path in files_to_check:
    if os.path.exists(file_path):
        size_kb = os.path.getsize(file_path) / 1024
        print(f"✓ {file_path} ({size_kb:.2f} KB)")
    else:
        print(f"✗ Missing: {file_path}")

Notebook 2 output files:

✓ ./rmit_assessment_support_chunks.csv (98.44 KB)
✓ ./rmit_assessment_support.faiss (106.54 KB)
✓ ./rmit_assessment_support_metadata.pkl (88.98 KB)
✓ ./rmit_retrieval_initial_evaluation.csv (0.58 KB)
